In [6]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error
from category_encoders import TargetEncoder
import optuna
import pickle

c:\Users\CYTech Student\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:

# Rendre Optuna silencieux
optuna.logging.set_verbosity(optuna.logging.WARNING)

# df = pd.read_csv('train.csv')
# categorical_features = ['type_contrat', 'utilisation', 'sex_conducteur1', 'essence_vehicule'] 
# numerical_features = ['bonus', 'anciennete_vehicule', 'age_conducteur1','puissance_age','ratio_poids_puissance', 'log_prix_vehicule', 'cp_dep', 'ratio_permis_age','din_vehicule']

def data():
    df = pd.read_csv('train.csv')
    return df

def cat():
    categorical_features = ['type_contrat', 'utilisation', 'sex_conducteur1', 'essence_vehicule'] 
    numerical_features = ['bonus', 'anciennete_vehicule', 'age_conducteur1','puissance_age','ratio_poids_puissance', 'log_prix_vehicule', 'cp_dep', 'ratio_permis_age','din_vehicule']
    return categorical_features, numerical_features
#####################
# --- FONCTION DE NETTOYAGE ---
def clean_data():
    df = data()
    df['montant_sinistre'] = df['montant_sinistre'].fillna(0)
    df['montant_sinistre'] = df['montant_sinistre'].where(df['montant_sinistre'] >= 0, 0)
    df = df[df['poids_vehicule'] > 0]

    return df

# --- 1. FONCTION DE FEATURE ENGINEERING ---
def apply_feature_engineering(df):
    df['ratio_poids_puissance'] = df['poids_vehicule'] / (df['cylindre_vehicule'] + 1)
    df['log_prix_vehicule'] = np.log1p(df['prix_vehicule'])     
    df['cp_dep'] = df['code_postal'].astype(str).str.zfill(5).str[:2]
    df['is_sportive'] = ((df['din_vehicule'] > 150) | (df['vitesse_vehicule'] > 200)).astype(int)
    df["ratio_permis_age"] = df["anciennete_permis1"] / (df["age_conducteur1"] + 1)
    df["puissance_age"] = df["din_vehicule"] * df["age_conducteur1"]
    return df

# Application
# df = clean_data(df)
# df = apply_feature_engineering(df)

def df_clean_fe():
    df = clean_data()
    df = apply_feature_engineering(df)
    return df


# --- b. Préparation ---

def prepare_data():
    df = df_clean_fe()
    categorical_features, numerical_features = cat()
    X = df[numerical_features + categorical_features]
    y_freq = df['nombre_sinistres']
    X_train, X_test, y_train, y_test = train_test_split(X, y_freq, test_size=0.2, random_state=8)
    encoder = TargetEncoder(cols=['cp_dep'])
    X_train = encoder.fit_transform(X_train, y_train)
    X_test = encoder.transform(X_test)
    return X_train, X_test, y_train, y_test,categorical_features

In [16]:
# # =====================================================================
# # --- 2. MODÈLE FRÉQUENCE (Probabilité de sinistre) ---
# # =====================================================================


# --- b. Préparation ---
##optuna
def objective_freq(trial):
    df = df_clean_fe()
    categorical_features, numerical_features = cat()
    X = df[numerical_features + categorical_features]
    y_freq = df['nombre_sinistres']
    X_train, X_test, y_train, y_test,categorical_features = prepare_data()
    params = {
        "iterations": trial.suggest_int("iterations", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "eval_metric": "AUC",
        "verbose": False
    }
    X_tr_opt, X_val_opt, y_tr_opt, y_val_opt = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    model = CatBoostClassifier(**params, auto_class_weights="Balanced")
    model.fit(X_tr_opt, y_tr_opt, cat_features=categorical_features, eval_set=(X_val_opt, y_val_opt), early_stopping_rounds=20)    
    preds_proba = model.predict_proba(X_test)[:, 1]
    auc_val = roc_auc_score(y_test, preds_proba)
    
    print(f'Freq parameters: {params}, AUC: {auc_val:.4f}')
    return auc_val

########
def train_model_freq():
    df = df_clean_fe()
    categorical_features, numerical_features = cat()
    X = df[numerical_features + categorical_features]
    y_freq = df['nombre_sinistres']
    X_train, X_test, y_train, y_test,categorical_features = prepare_data()
    print("--- Recherche des hyperparamètres FRÉQUENCE (Maximisation AUC) ---")
    study_freq = optuna.create_study(direction="maximize")
    study_freq.optimize(objective_freq, n_trials=100) 

    # Entraînement final Fréquence
    print(f"\nBest Freq parameters: {study_freq.best_params}")
    model_freq = CatBoostClassifier(**study_freq.best_params, auto_class_weights="Balanced", eval_metric="AUC", verbose=False)
    model_freq.fit(X_train, y_train, cat_features=categorical_features, eval_set=(X_test, y_test))
    #sauvegarde en pickle
    with open('model_freq.pkl', 'wb') as f:
        pickle.dump(model_freq, f)

    return model_freq
# =====================================================================
# --- 3. MODÈLE SÉVÉRITÉ (Montant moyen) ---
# =====================================================================
def model_severite():
    df = df_clean_fe()
    encoder = TargetEncoder(cols=['cp_dep'])
    categorical_features, numerical_features = cat()
    df_claims = df.copy() 
    X_sev = df_claims[numerical_features + categorical_features]
    y_sev = df_claims['montant_sinistre']
    X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(X_sev, y_sev, test_size=0.2, random_state=8)
    X_train_sev = encoder.fit_transform(X_train_sev, y_train_sev)
    X_test_sev = encoder.transform(X_test_sev)
    return X_train_sev, X_test_sev, y_train_sev, y_test_sev,categorical_features

def objective_sev(trial):
    df = df_clean_fe()
    categorical_features, numerical_features = cat()
    X_train_sev, X_test_sev, y_train_sev, y_test_sev,categorical_features=model_severite(df)         
    params = {
        "iterations": trial.suggest_int("iterations", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "objective": 'Tweedie:variance_power=1.5', 
        "verbose": False
    }
    
    model = CatBoostRegressor(**params) # Pas de class_weights pour une régression
    X_tr_sev_opt, X_val_sev_opt, y_tr_sev_opt, y_val_sev_opt = train_test_split(
        X_train_sev, y_train_sev, test_size=0.2, random_state=42
    )
    model.fit(X_tr_sev_opt, y_tr_sev_opt, cat_features=categorical_features, eval_set=(X_val_sev_opt, y_val_sev_opt), early_stopping_rounds=20)    
    preds = model.predict(X_test_sev)
    mse_val = mean_squared_error(y_test_sev, preds)
    
    print(f'Sev parameters: {params}, MSE: {mse_val:.2f}')
    return mse_val


def train_model_sev():
    df = df_clean_fe()
    categorical_features, numerical_features = cat()
    X = df[numerical_features + categorical_features]
    y_freq = df['nombre_sinistres']
    X_train_sev, X_test_sev, y_train_sev, y_test_sev,categorical_features = prepare_data()
    print("\n--- Recherche des hyperparamètres SÉVÉRITÉ (Minimisation MSE) ---")
    study_sev = optuna.create_study(direction="minimize")
    study_sev.optimize(objective_sev, n_trials=100) # Ajuster si besoin

    # Entraînement final Sévérité
    print(f"\nBest Sev parameters: {study_sev.best_params}")
    # On s'assure de réinjecter l'objectif Tweedie qui n'est pas dans best_params par défaut s'il n'était pas dans l'espace de recherche suggest
    best_sev_params = study_sev.best_params
    best_sev_params['objective'] = 'Tweedie:variance_power=1.5'

    model_sev = CatBoostRegressor(**best_sev_params, verbose=False)
    model_sev.fit(X_train_sev, y_train_sev, cat_features=categorical_features, eval_set=(X_test_sev, y_test_sev))
    with open('model_sev.pkl', 'wb') as f:
        pickle.dump(model_sev, f)
    print("Modèles exportés avec succès !")
    return model_sev


# =====================================================================
# --- MÉTRIQUES FINALES ---
# =====================================================================
def predict_final():
    df = df_clean_fe()
    categorical_features, numerical_features = cat()
    X = df[numerical_features + categorical_features]
    y_freq = df['nombre_sinistres']
    X_test, y_test, X_test_sev, y_test_sev,categorical_features = prepare_data()
    model_freq = train_model_freq()
    model_sev = train_model_sev()
    encoder = TargetEncoder(cols=['cp_dep'])
    y_pred_freq_proba = model_freq.predict_proba(X_test)[:, 1]
    auc_final = roc_auc_score(y_test, y_pred_freq_proba)
    print(f"\nROC AUC final for the frequency model: {auc_final:.4f}")

    y_pred_sev = model_sev.predict(X_test_sev)
    mse_sev_final = mean_squared_error(y_test_sev, y_pred_sev)
    print(f"Mean Squared Error final for the severity model: {mse_sev_final:.2f}")


    # --- 4. CALCUL DU COÛT ESPÉRÉ SUR TOUT LE DATASET ---
    X_encoded = encoder.transform(X)

    df['probabilite'] = model_freq.predict_proba(X_encoded)[:, 1]
    df['montant_potentiel'] = model_sev.predict(X_encoded)
    df['pred'] = df['probabilite'] * df['montant_potentiel']

    print("\nAperçu des prédictions :")
    print(df[['probabilite', 'montant_potentiel', 'pred']].head())

In [17]:
predict_final()

--- Recherche des hyperparamètres FRÉQUENCE (Maximisation AUC) ---
Freq parameters: {'iterations': 326, 'learning_rate': 0.014854878266801473, 'depth': 5, 'l2_leaf_reg': 3.0997800233691586, 'random_strength': 3.971290584927821, 'bagging_temperature': 0.32107625884316116, 'border_count': 71, 'eval_metric': 'AUC', 'verbose': False}, AUC: 0.6523
Freq parameters: {'iterations': 435, 'learning_rate': 0.06038600981654309, 'depth': 6, 'l2_leaf_reg': 2.898565819617009, 'random_strength': 0.33248075910797814, 'bagging_temperature': 0.17941377923605595, 'border_count': 53, 'eval_metric': 'AUC', 'verbose': False}, AUC: 0.6460
Freq parameters: {'iterations': 261, 'learning_rate': 0.09165071834868138, 'depth': 4, 'l2_leaf_reg': 8.833291770640876, 'random_strength': 4.582028695046904e-09, 'bagging_temperature': 0.5170693768561175, 'border_count': 77, 'eval_metric': 'AUC', 'verbose': False}, AUC: 0.6475


[W 2026-03-17 12:03:55,764] Trial 3 failed with parameters: {'iterations': 213, 'learning_rate': 0.017940235535364536, 'depth': 9, 'l2_leaf_reg': 8.061178582089905, 'random_strength': 1.7465084336544146e-06, 'bagging_temperature': 0.11239484819656642, 'border_count': 227} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "c:\Users\CYTech Student\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\CYTech Student\AppData\Local\Temp\ipykernel_34724\314639917.py", line 29, in objective_freq
    model.fit(X_tr_opt, y_tr_opt, cat_features=categorical_features, eval_set=(X_val_opt, y_val_opt), early_stopping_rounds=20)
  File "c:\Users\CYTech Student\AppData\Local\Programs\Python\Python311\Lib\site-packages\catboost\core.py", line 5245, in fit
    self._fit(X, y, cat_features, text_features, embedding_feat

KeyboardInterrupt: 

In [ ]:
# 1. Charger le nouveau fichier
nouveau_df = pd.read_csv('test.csv')

# --- CORRECTIF 1 : Appliquer le nettoyage de base (sans toucher à la cible) ---
# On utilise les moyennes calculées sur ton dataset d'entraînement (df)
nouveau_df['poids_vehicule'] = nouveau_df['poids_vehicule'].replace(0, df['poids_vehicule'].mean())
nouveau_df['cylindre_vehicule'] = nouveau_df['cylindre_vehicule'].replace(0, df['cylindre_vehicule'].mean())

nouveau_df = apply_feature_engineering(nouveau_df)  

# 3. Sélectionner les colonnes utilisées par le modèle
X_nouveau = nouveau_df[numerical_features + categorical_features]

# --- CORRECTIF 2 : ENCODER LE JEU DE TEST ---
# C'est ici qu'on transforme la Corse ("2A"/"2B") et les autres départements en nombres
X_nouveau_encoded = encoder.transform(X_nouveau)

# 4. Appliquer les prédictions (sur les données ENCODÉES)
# Probabilité de sinistre (Modèle 1)
nouveau_df['probabilite'] = model_freq.predict_proba(X_nouveau_encoded)[:, 1]

# Montant estimé si sinistre il y a (Modèle 2)
nouveau_df['montant_potentiel'] = model_sev.predict(X_nouveau_encoded)

# Coût espéré (Prime pure)
nouveau_df['pred'] = nouveau_df['probabilite'] * nouveau_df['montant_potentiel']

# 5. Sauvegarder les résultats complets
nouveau_df.to_csv('resultats_predictions.csv', index=False)
print("Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'")

# 6. Créer le fichier final pour soumission
pp = pd.read_csv('resultats_predictions.csv')

# Assure-toi que la colonne s'appelle bien 'index' dans ton test.csv d'origine
df_final = pp[['index', 'pred']]

# Sauvegarder le résultat final
df_final.to_csv('resultat_final.csv', index=False)
print('Le fichier filtré a été sauvegardé sous le nom resultat_final.csv')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Calcul de l'AUC et de la courbe ROC
y_probs = model_freq.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_probs)
fpr, tpr, thresholds = roc_curve(y_test, y_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'CatBoost (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Taux de Faux Positifs')
plt.ylabel('Taux de Vrais Positifs')
plt.title('Courbe ROC - Modèle Fréquence')
plt.legend()
plt.show()

# 2. Matrice de Confusion (Seuil optimisé)
y_pred_custom = (y_probs > 0.2).astype(int)
cm = confusion_matrix(y_test, y_pred_custom)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Matrice de Confusion (Seuil 0.2)')
plt.show()

# 3. Importance des variables
feature_importance = model_freq.get_feature_importance()
sorted_idx = np.argsort(feature_importance)
feature_names = np.array(X_test.columns[sorted_idx])

plt.figure(figsize=(10, 8))
plt.barh(feature_names, feature_importance[sorted_idx])
plt.xlabel("CatBoost Feature Importance")
plt.show()


feature_importance_sev = model_sev.get_feature_importance()
sorted_idx = np.argsort(feature_importance_sev)
feature_names = np.array(X_test.columns[sorted_idx])
plt.figure(figsize=(10, 8))
plt.barh(feature_names, feature_importance_sev[sorted_idx])
plt.xlabel("CatBoost Feature Importance")
plt.show()





# 4. Courbe ROC pour le modèle de Sévérité
y_probs_sev = model_sev.predict(X_test_sev)
mse_score_sev = mean_squared_error(y_test_sev, y_probs_sev)  # Calcul du MSE pour le modèle de Sévérité
fpr_sev, tpr_sev, thresholds_sev = roc_curve(y_test_sev > 0, y_probs_sev)

plt.figure(figsize=(8, 6))
plt.plot(fpr_sev, tpr_sev, label=f'CatBoost (MSE = {mse_score_sev:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Taux de Faux Positifs')
plt.ylabel('Taux de Vrais Positifs')
plt.title('Courbe ROC - Modèle Sévérité')
plt.legend()
plt.show()

# 5. Matrice de Confusion pour le modèle de Sévérité (Seuil optimisé)
y_pred_custom_sev = (y_probs_sev > 0.2).astype(int)
cm_sev = confusion_matrix(y_test_sev > 0, y_pred_custom_sev)
disp_sev = ConfusionMatrixDisplay(confusion_matrix=cm_sev)
disp_sev.plot(cmap='Blues')
plt.title('Matrice de Confusion (Seuil 0.2) - Modèle Sévérité')
plt.show()

# 6. Importance des variables pour le modèle de Sévérité
sorted_idx_sev = np.argsort(feature_importance_sev)
feature_names_sev = np.array(X_test_sev.columns[sorted_idx_sev])

plt.figure(figsize=(10, 8))
plt.barh(feature_names_sev, feature_importance_sev[sorted_idx_sev])
plt.xlabel("CatBoost Feature Importance - Modèle Sévérité")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

# =====================================================================
# --- 5. GÉNÉRATION DES GRAPHIQUES POUR LA PRÉSENTATION ---
# =====================================================================
sns.set_theme(style="whitegrid") # Style épuré pour les slides

# ---------------------------------------------------------
# GRAPHIQUE 1 : Courbe ROC (Modèle de Fréquence)
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
fpr, tpr, thresholds = roc_curve(y_test, y_pred_freq_proba)

plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {auc_final:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aléatoire')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
plt.title('Performance du Modèle de Fréquence (Courbe ROC)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.tight_layout()
plt.savefig('plot_roc_frequence.png', dpi=300) # Sauvegarde l'image en haute qualité
plt.show()

# ---------------------------------------------------------
# GRAPHIQUES 2 & 3 : Sévérité (Réel vs Prédit & Feature Importance)
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphe de gauche : Réel vs Prédit
axes[0].scatter(y_test_sev, y_pred_sev, alpha=0.4, color='teal', edgecolors='w')
# Ligne diagonale parfaite (y = x)
max_val = max(y_test_sev.max(), y_pred_sev.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Prédiction Parfaite')
axes[0].set_title('Sévérité : Montants Réels vs Prédits', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Montant Réel du Sinistre (€)', fontsize=12)
axes[0].set_ylabel('Montant Prédit par le Modèle (€)', fontsize=12)
axes[0].legend()

# Graphe de droite : Importance des Variables (Top 10)
# CatBoost possède une fonction native très pratique pour ça
feature_importances = model_sev.get_feature_importance()
features = X_sev.columns
importance_df = pd.DataFrame({
    'Variable': features, 
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False).head(10) # On garde le Top 10

sns.barplot(x='Importance', y='Variable', data=importance_df, ax=axes[1], palette='viridis')
axes[1].set_title('Top 10 des Variables (Modèle Sévérité)', fontsize=14, fontweight='bold')
axes[1].set_xlabel("Niveau d'importance", fontsize=12)
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('plot_severite_analyse.png', dpi=300) # Sauvegarde
plt.show()

In [ ]:
import seaborn as sns

test = pd.read_csv('resultat_final.csv')
print(test.head())

print("Moyenne des prédictions :", test['pred'].mean())
print("Somme des prédictions :", test['pred'].sum())
print("Somme des montants de sinistre :", df['montant_sinistre'].sum())
sns.histplot(test['pred'], kde=True, bins=30, color='blue')
plt.title('Distribution des Primes')
plt.xlabel('Primes')
plt.ylabel('Fréquence')
plt.show()

